# Marketing KPI Prediction: Lead Conversion
### Original example by Danny Barren: *ML for Business Insights, Examples*

**The KPI:** *lead-to-customer conversion rate.*
**The business question:** *Which new leads are most likely to become paying customers, so sales spends its limited time on them first?*

Most teams work leads first-come, first-served. This notebook scores every lead with a **probability of converting** using a **HistGradientBoostingClassifier** (with a logistic regression as a readable benchmark), then answers the questions marketing and sales actually ask:

1. How much better than random is our lead ranking? (ROC-AUC, PR-AUC, **lift**)
2. If sales can only call the top X% of leads, **what share of conversions do we capture?**
3. Where should we draw the line, given the **value of a conversion** and the **cost of a sales touch**?
4. Which lead behaviours and sources **signal buying intent**?

**How to use it:** Edit only the **CONFIG** cell, then run top to bottom.
If `CSV_PATH = None`, a **synthetic** lead dataset is created for the demo.

In [ ]:
# ============================================================
# IMPORTS (scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, loguniform

from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_validate, cross_val_predict, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================
CSV_PATH = None                     # e.g. "/content/leads.csv"; None = synthetic demo

ID_COLUMN = "lead_id"
TARGET_COLUMN = "converted"         # did the lead become a paying customer?
POSITIVE_VALUE = 1                  # value meaning "converted" (1, "Yes", "won", ...)

# Only use what you know BEFORE sales works the lead. Leakage examples to EXCLUDE:
# "deal_stage", "contract_value", "sales_calls_made", "closed_date".
NUMERIC_COLUMNS = ["pages_viewed", "email_opens", "email_clicks", "days_since_first_touch",
                   "company_employees", "form_fields_completed", "pricing_page_visits"]
CATEGORICAL_COLUMNS = ["lead_source", "industry", "webinar_attended", "region"]
DROP_COLUMNS = []

# --- Business numbers ------------------------------------------------
VALUE_PER_CONVERSION = 2_000        # first-year gross margin from one new customer
COST_PER_SALES_TOUCH = 100          # rep time + tools to properly work one lead
SALES_LIFT_RATE = 0.30              # share of would-be converters a good sales touch actually wins
                                    # (an untouched lead converts less often or goes to a competitor)
SALES_CAPACITY_SHARE = 0.20         # share of incoming leads the team can properly work (None = no cap)

MAIN_METRIC = "average_precision"   # PR-AUC: best when conversion is rare
DATA_SIZE_MODE = "auto"
TEST_SIZE = 0.25
RANDOM_STATE = 42

In [ ]:
# ============================================================
# LOAD DATA (or build synthetic leads)
# ============================================================
def make_demo_leads(n=800, seed=RANDOM_STATE):
    """Synthetic B2B leads. Story: pricing-page visits, webinar attendance, referrals
    and email clicks signal intent; conversion fades the longer a lead goes cold;
    mid-size companies convert best (a non-linear pattern trees pick up easily)."""
    rng = np.random.default_rng(seed)
    source = rng.choice(["paid_search", "organic", "referral", "trade_show", "social"], n,
                        p=[0.30, 0.25, 0.12, 0.13, 0.20])
    industry = rng.choice(["retail", "manufacturing", "services", "healthcare", "tech"], n)
    webinar = rng.choice(["yes", "no"], n, p=[0.25, 0.75])
    region = rng.choice(["northeast", "south", "midwest", "west"], n)
    pages = rng.poisson(6, n) + 1
    opens = rng.poisson(4, n)
    clicks = rng.binomial(opens, 0.3)
    days = rng.integers(1, 120, n)
    employees = np.round(rng.lognormal(np.log(80), 1.1, n))
    fields = rng.integers(2, 10, n)
    pricing = rng.poisson(0.8 + 0.1 * pages, n)
    size_sweet_spot = np.exp(-((np.log(employees) - np.log(150)) ** 2) / 1.5)
    logit = (-4.3 + 0.65 * np.minimum(pricing, 4) + 0.9 * (webinar == "yes") + 1.0 * (source == "referral")
             + 0.5 * (source == "trade_show") - 0.4 * (source == "social") + 0.25 * clicks
             + 0.06 * pages - 0.015 * days + 1.0 * size_sweet_spot + 0.08 * fields
             + 0.3 * (industry == "tech"))
    conv = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame({"lead_id": np.arange(5_001, 5_001 + n), "pages_viewed": pages, "email_opens": opens,
                       "email_clicks": clicks, "days_since_first_touch": days, "company_employees": employees,
                       "form_fields_completed": fields, "pricing_page_visits": pricing,
                       "lead_source": source, "industry": industry, "webinar_attended": webinar,
                       "region": region, "converted": conv})
    df.loc[rng.random(n) < 0.05, "company_employees"] = np.nan   # firmographic gaps are common
    df.loc[rng.random(n) < 0.03, "industry"] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo_leads()
    print("Using SYNTHETIC demo data (CSV_PATH is None).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"{len(df):,} leads x {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK
# ============================================================
needed = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in needed if c not in df.columns]
assert not missing, f"These CONFIG columns are not in your file: {missing}"

y_all = (df[TARGET_COLUMN] == POSITIVE_VALUE).astype(int)
assert y_all.nunique() == 2, "Need both converted and non-converted leads. Check POSITIVE_VALUE."
FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X_all = df[FEATURES].copy()
for c in NUMERIC_COLUMNS:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce").astype(float)
for c in CATEGORICAL_COLUMNS:
    X_all[c] = X_all[c].astype("string").astype(object).where(X_all[c].notna(), np.nan)
print(f"Conversion rate: {y_all.mean():.1%}  ({y_all.sum():,} of {len(y_all):,} leads)")

## 1. Light EDA: which sources and behaviours convert?

In [ ]:
display(pd.DataFrame({"missing_%": X_all.isna().mean() * 100, "unique": X_all.nunique()}).round(1))

fig, axes = plt.subplots(1, len(CATEGORICAL_COLUMNS), figsize=(4.5 * len(CATEGORICAL_COLUMNS), 3.6))
for ax, col in zip(np.atleast_1d(axes), CATEGORICAL_COLUMNS):
    y_all.groupby(X_all[col].fillna("missing")).mean().sort_values().plot.barh(ax=ax, color="seagreen")
    ax.axvline(y_all.mean(), color="k", ls="--", lw=1); ax.set_title(f"Conversion by {col}")
plt.tight_layout(); plt.show()

# Leakage sniff test: a single column that predicts conversion almost perfectly is suspicious
sniff = {c: roc_auc_score(y_all, X_all[c].fillna(X_all[c].median())) for c in NUMERIC_COLUMNS}
sniff = pd.Series({k: max(v, 1 - v) for k, v in sniff.items()}).sort_values(ascending=False)
display(sniff.to_frame("single_column_AUC"))
print("Possible leakage:", sniff[sniff > 0.90].index.tolist() or "none found")

## 2. Split, validation plan, and two leakage-safe pipelines
- **Logistic regression** (benchmark, easy to read): scale numbers, one-hot encode categories
- **HistGradientBoosting** (main model): handles curved patterns and interactions; categories passed as integer codes

In [ ]:
size_mode = DATA_SIZE_MODE if DATA_SIZE_MODE != "auto" else ("small" if len(df) < 1_000 else "large")
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE,
                                                    stratify=y_all, random_state=RANDOM_STATE)
cv = (RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if size_mode == "small"
      else StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))

logit_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_COLUMNS),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL_COLUMNS),
    ], verbose_feature_names_out=False)),
    ("model", LogisticRegression(C=0.1, max_iter=2_000)),
])

hgb_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("codes", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
         CATEGORICAL_COLUMNS),
        ("num", "passthrough", NUMERIC_COLUMNS),          # HGB handles missing numbers natively
    ], verbose_feature_names_out=False)),
    ("model", HistGradientBoostingClassifier(
        categorical_features=[True] * len(CATEGORICAL_COLUMNS) + [False] * len(NUMERIC_COLUMNS),
        early_stopping=(size_mode == "large"), random_state=RANDOM_STATE)),
])
print(f"Data size mode: {size_mode} | train {len(X_train):,}, test {len(X_test):,} leads")

## 3. Baseline vs. both models (cross-validated on training data only)

In [ ]:
scoring = {"roc_auc": "roc_auc", "average_precision": "average_precision"}
def cv_summary(name, model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
    return {"model": name, "ROC_AUC": r["test_roc_auc"].mean(), "PR_AUC": r["test_average_precision"].mean(),
            "PR_AUC_spread": r["test_average_precision"].std(),
            "train_minus_valid_PR_AUC": r["train_average_precision"].mean() - r["test_average_precision"].mean()}

display(pd.DataFrame([cv_summary("Dummy (always the average)", DummyClassifier(strategy="prior")),
                      cv_summary("Logistic regression", logit_pipe),
                      cv_summary("HistGradientBoosting (defaults)", hgb_pipe)]))
print("A large train-minus-valid gap means the model is memorising. Tuning below reins that in.")

## 4. Light tuning of the boosting model
We try 25 random settings with **RandomizedSearchCV**. With small data, careful settings (low learning rate, few leaves, big minimum leaf size) usually win.
**Simplicity rule:** among settings within one standard error of the best, take the one with the fewest leaves, then the biggest leaf size.

In [ ]:
param_dist = {"model__learning_rate": loguniform(0.01, 0.2),
              "model__max_iter": randint(50, 300),
              "model__max_leaf_nodes": randint(3, 20),
              "model__min_samples_leaf": randint(10, 60),
              "model__l2_regularization": loguniform(1e-3, 10)}
search = RandomizedSearchCV(hgb_pipe, param_dist, n_iter=25, cv=cv, scoring=MAIN_METRIC,
                            random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)

res = pd.DataFrame(search.cv_results_)
best = res.loc[res["rank_test_score"].idxmin()]
se = best["std_test_score"] / np.sqrt(cv.get_n_splits())
close = res[res["mean_test_score"] >= best["mean_test_score"] - se].sort_values(
    ["param_model__max_leaf_nodes", "param_model__min_samples_leaf"], ascending=[True, False])
chosen_params = close.iloc[0]["params"]
print(f"Best CV PR-AUC {best['mean_test_score']:.3f} | chosen (simplest within 1 SE) {close.iloc[0]['mean_test_score']:.3f}")
print("Chosen:", {k.replace("model__", ""): (round(float(v), 4) if isinstance(v, (float, np.floating)) else v)
                  for k, v in chosen_params.items()})

hgb_tuned = hgb_pipe.set_params(**chosen_params)
# Champion = whichever model has the better cross-validated PR-AUC (ties go to the simpler logistic)
logit_cv = cross_validate(logit_pipe, X_train, y_train, cv=cv, scoring=MAIN_METRIC)["test_score"].mean()
champion_name = "HistGradientBoosting" if close.iloc[0]["mean_test_score"] > logit_cv + 0.005 else "Logistic regression"
champion = (hgb_tuned if champion_name == "HistGradientBoosting" else logit_pipe).fit(X_train, y_train)
logit_pipe.fit(X_train, y_train)     # also fit the benchmark: we use it for readable coefficients later
print(f"Champion model: {champion_name}")

## 5. The honest exam: test set, lift, and capture curve
**Lift** answers: *"Leads in the top 10% by score convert how many times more often than average?"*
The **capture curve** answers: *"If we work the top X% of leads, what share of all conversions do we reach?"*

In [ ]:
p_test = champion.predict_proba(X_test)[:, 1]
print(f"Test ROC-AUC {roc_auc_score(y_test, p_test):.3f} (coin flip 0.5) | "
      f"PR-AUC {average_precision_score(y_test, p_test):.3f} (random {y_test.mean():.3f})")

ranked = pd.DataFrame({"p": p_test, "y": y_test.values}).sort_values("p", ascending=False).reset_index(drop=True)
ranked["decile"] = pd.qcut(ranked.index, 10, labels=range(1, 11))
lift = ranked.groupby("decile", observed=True).agg(leads=("y", "size"), conversions=("y", "sum"), conv_rate=("y", "mean"))
lift["lift_vs_avg"] = lift["conv_rate"] / ranked["y"].mean()
lift["cum_share_of_conversions"] = lift["conversions"].cumsum() / ranked["y"].sum()
display(lift.style.format({"conv_rate": "{:.1%}", "lift_vs_avg": "{:.1f}x", "cum_share_of_conversions": "{:.0%}"}))

share_leads = np.arange(1, len(ranked) + 1) / len(ranked)
plt.figure(figsize=(5.5, 4))
plt.plot(share_leads, ranked["y"].cumsum() / ranked["y"].sum(), label="model ranking")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="random order")
plt.xlabel("share of leads worked (highest score first)"); plt.ylabel("share of conversions captured")
plt.title("Capture curve"); plt.legend(); plt.show()

## 6. Where to draw the line: expected value with sales capacity
For each cut-off we estimate the extra margin from working those leads:

> value = (converters worked × `SALES_LIFT_RATE` × `VALUE_PER_CONVERSION`) − (leads worked × `COST_PER_SALES_TOUCH`)

Chosen on **out-of-fold** training predictions, then checked once on the test set. If `SALES_CAPACITY_SHARE` is set, we also show the value of simply working the top share of leads the team can handle.

In [ ]:
p_oof = cross_val_predict(champion, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                          method="predict_proba")[:, 1]
def value_at(y_true, p, t):
    work = p >= t
    return (work & (y_true == 1)).sum() * SALES_LIFT_RATE * VALUE_PER_CONVERSION - work.sum() * COST_PER_SALES_TOUCH

rows = []
for t in np.round(np.arange(0.05, 0.71, 0.05), 2):
    work = p_oof >= t
    rows.append({"threshold": t, "worked_%": work.mean() * 100, "precision": y_train.values[work].mean() if work.any() else 0,
                 "recall": (work & (y_train.values == 1)).sum() / y_train.sum(),
                 "value_per_1000_leads": value_at(y_train.values, p_oof, t) / len(y_train) * 1000})
tbl = pd.DataFrame(rows)
best_t = tbl.loc[tbl["value_per_1000_leads"].idxmax(), "threshold"]
display(tbl.style.format({"worked_%": "{:.1f}", "precision": "{:.0%}", "recall": "{:.0%}",
                          "value_per_1000_leads": "${:,.0f}"}).highlight_max("value_per_1000_leads", color="#c6efce"))
be = COST_PER_SALES_TOUCH / (SALES_LIFT_RATE * VALUE_PER_CONVERSION)
print(f"Best threshold (out-of-fold): {best_t:.2f}. Break-even: a lead is worth working if P(convert) > {be:.2f}")

test_value = value_at(y_test.values, p_test, best_t)
print(f"Test check: working {int((p_test >= best_t).sum())} of {len(p_test)} leads -> ${test_value:,.0f} expected value "
      f"(${test_value / len(p_test) * 1000:,.0f} per 1,000 leads)")
if SALES_CAPACITY_SHARE:
    cap = int(round(SALES_CAPACITY_SHARE * len(p_test)))
    top = np.argsort(-p_test)[:cap]
    cap_value = y_test.values[top].sum() * SALES_LIFT_RATE * VALUE_PER_CONVERSION - cap * COST_PER_SALES_TOUCH
    print(f"With capacity {SALES_CAPACITY_SHARE:.0%} of leads: work the top {cap} test leads -> ${cap_value:,.0f}, "
          f"capturing {y_test.values[top].sum() / y_test.sum():.0%} of conversions.")

## 7. What signals buying intent? (plain-language interpretation)
- **Permutation importance** on the champion: how much the ranking quality drops when one input is scrambled.
- **Logistic odds multipliers** from the benchmark: the *direction* of each effect, in words.

> Effects for small groups (a category with only a few dozen leads) can be noise on small data. Trust the drivers that show up in **both** views.

In [ ]:
perm = permutation_importance(champion, X_test, y_test, scoring=MAIN_METRIC, n_repeats=20,
                              random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 4.5), color="teal"); plt.title(f"Permutation importance ({champion_name})")
plt.tight_layout(); plt.show()

coefs = pd.Series(logit_pipe.named_steps["model"].coef_[0], index=logit_pipe.named_steps["prep"].get_feature_names_out())
odds = np.exp(coefs).sort_values()
sd = X_train[NUMERIC_COLUMNS].std()
print("In plain English (from the logistic benchmark, effects holding other inputs fixed):")
for feat in list(odds.index[::-1][:4]) + list(odds.index[:2]):
    change = (odds[feat] - 1) * 100
    if feat in NUMERIC_COLUMNS:
        print(f"  • +{sd[feat]:,.1f} {feat} (one typical step) → {change:+.0f}% odds of converting")
    else:
        print(f"  • {feat} → {change:+.0f}% odds of converting vs. the average lead")

## 8. What to do next: business recommendations
**Sales**
- Sort the lead queue by score every morning; work the top of the list within 24 hours (intent fades with time).
- Set the cut-off from the expected-value table, or simply work the top `SALES_CAPACITY_SHARE` of leads when the team is full.

**Marketing**
- Shift budget toward the sources and behaviours with the strongest positive effects (in the demo: referrals, webinars, pricing-page visits).
- Nurture low-score leads with automated email instead of rep time, and re-score them when they show new intent signals.

**Prove it:** for one month, randomly split new leads, with half worked by score and half business-as-usual. Compare conversion rate and revenue per rep hour.

**Keep it healthy:** watch the top-decile conversion rate monthly; retrain each quarter or after a big campaign or website change.

**Small vs. large data**
- *Small (< 1,000 leads):* the logistic benchmark is often as good as boosting. Keep whichever is simpler when scores are close.
- *Large (10k+ leads):* boosting with early stopping usually pulls ahead; a single 5-fold CV is enough.

In [ ]:
# Optional: export a scored lead list for the sales team and save the model
out = X_test.copy()
if ID_COLUMN in df.columns:
    out.insert(0, ID_COLUMN, df.loc[X_test.index, ID_COLUMN])
out["conversion_probability"] = p_test.round(3)
out["action"] = np.where(p_test >= best_t, "Sales: call within 24h", "Marketing: nurture")
out.sort_values("conversion_probability", ascending=False).to_csv("scored_leads.csv", index=False)
import joblib
joblib.dump({"model": champion, "threshold": float(best_t), "features": FEATURES}, "lead_conversion_model.joblib")
display(out.sort_values("conversion_probability", ascending=False).head(10))
print("Saved scored_leads.csv and lead_conversion_model.joblib")